In [3]:
import json
from collections import defaultdict
from typing import Callable, Optional, Dict, Any

def find_duplicates_in_jsonl(
    file_path: str,
    key: Optional[str] = None,
    key_fn: Optional[Callable[[Dict[str, Any]], Any]] = None,
    return_locations: bool = True,
):
    """
    Scan a JSONL file for duplicates.

    Args:
        file_path: path to .jsonl
        key: if provided, use record[key] as the identity
        key_fn: custom function(record)->identity (overrides `key` if given)
        return_locations: if True, include line numbers for each duplicate

    Returns:
        duplicates: dict(identity -> {"count": int, "lines": [ints], "example": record})
                     If return_locations=False, values are just counts.
    """
    def canonicalize(obj: Any) -> str:
        # Make any JSON-serializable object hashable/comparable
        return json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(",", ":"))

    seen_lines = defaultdict(list)  # identity -> list of line numbers
    examples = {}                   # identity -> first seen record

    with open(file_path, "r", encoding="utf-8") as f:
        for ln, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError as e:
                # Skip bad lines but keep going
                # You can raise instead if you prefer strict behavior
                # print(f"JSON decode error on line {ln}: {e}")
                continue

            # Decide identity
            if key_fn is not None:
                ident_raw = key_fn(rec)
            elif key is not None:
                ident_raw = rec.get(key, None)
            else:
                ident_raw = rec  # whole-record comparison

            # Canonicalize to make it hashable/comparable
            ident = canonicalize(ident_raw)

            seen_lines[ident].append(ln)
            if ident not in examples:
                examples[ident] = rec

    # Build duplicates dict
    if return_locations:
        duplicates = {
            ident: {
                "count": len(lines),
                "lines": lines,
                "example": examples[ident],
            }
            for ident, lines in seen_lines.items()
            if len(lines) > 1
        }
    else:
        duplicates = {
            ident: len(lines)
            for ident, lines in seen_lines.items()
            if len(lines) > 1
        }

    return duplicates

# Examples:
# 1) whole-record duplicates
# dups = find_duplicates_in_jsonl("../src/results/meta-llama_Llama-3.2-3B_scores.jsonl")

# 2) duplicates by "id" field
# dups = find_duplicates_in_jsonl("data.jsonl", key="id")

# 3) duplicates by a compound identity (e.g., id + answer)
# dups = find_duplicates_in_jsonl("data.jsonl", key_fn=lambda r: {"id": r.get("id"), "answer": r.get("answer")})


In [ ]:
import json
from pathlib import Path

def remove_is_validation(input_path, output_path=None):
    """
    Remove 'is_validation' key from every JSON object in a JSONL file.

    Args:
        input_path (str | Path): Path to the original JSONL file.
        output_path (str | Path | None): Path to write the cleaned file.
            If None, overwrite the original file.
    """
    input_path = Path(input_path)
    if output_path is None:
        output_path = input_path
    else:
        output_path = Path(output_path)

    with input_path.open("r", encoding="utf-8") as infile, \
         output_path.open("w", encoding="utf-8") as outfile:
        for i, line in enumerate(infile, 1):
            line = line.strip()
            if not line:
                outfile.write("\n")
                continue
            try:
                obj = json.loads(line)
                if "is_validation" in obj:
                    del obj["is_validation"]
                json.dump(obj, outfile, ensure_ascii=False)
                outfile.write("\n")
            except json.JSONDecodeError as e:
                print(f"⚠️ Skipping invalid JSON at line {i}: {e}")
                outfile.write(line + "\n")

    print(f"✅ Finished. Saved cleaned file to {output_path}")

# Example usage:
remove_is_validation(
    "../results/meta-llama_Llama-3.2-3B_scores.jsonl",
    "../results/meta-llama_Llama-3.2-3B_scores_no_val.jsonl"
)


In [12]:
dups = find_duplicates_in_jsonl(path)

In [14]:
dups

{'{"answer":{"aliases":["Portogało","Republic of Portugal","PORTUGAL","Portekiz","Portugallu","O Papagaio","ISO 3166-1:PT","Portunga","Phu-to-ga","Potigal","Portûnga","Portugul","An Phortaingéil","Portugāle","Portugale","Portingale","Potiti","Portugali","Portugall","Portekîz","Bo Dao Nha","Portuguese Republic","Portogallo","Portugaul","Portogalo","Portyngal","Yn Phortiugal","Portugalio","Portugál","Portugual","Portuga","Portgual","Portugalsko","Portugaleje","Phû-tô-gâ","Portugalujo","Portugalija","Pertual","Pòtigal","Portugal","Bồ Đào Nha","Portugalska","República Portuguesa","Portiwgal","Portugalėjė","Portúgal","Portegal","An Phortaingeil","Republica Portuguesa"],"matched_wiki_entity_name":"","normalized_aliases":["portugul","portugallu","portugalska","pòtigal","portugaul","portugalujo","portuguese republic","iso 3166 1 pt","republic of portugal","portugalsko","portugual","bồ đào nha","portugall","portûnga","bo dao nha","phortaingeil","portugale","portugal","portugál","portugalėjė","p

In [35]:
path = "../results/meta-llama_Llama-3.2-3B_scores.jsonl"

In [36]:
deduped = deduplicate_jsonl(path,
                            output_path="../src/results/llama/meta-llama_Llama-3.2-3B_scores.jsonl")

✅ Deduplicated: kept 76499 / original 76499 unique by 'id'.


✅ Finished. Saved cleaned file to ../results/meta-llama_Llama-3.2-3B_scores_no_val.jsonl


In [6]:
import json

def count_zero_scores(file_path):
    """
    Count how many entries in a JSONL file have base_eval.score == 0.0.
    
    Args:
        file_path (str): Path to the .jsonl file.
    
    Returns:
        int: Number of entries with score 0.0.
    """
    zero_count = 0
    with open(file_path, "r", encoding="utf-8") as f:
        for ln, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"❌ JSON decode error on line {ln}: {e}")
                continue
            
            score = rec.get("base_eval", {}).get("score", None)
            if score == 0.0:
                zero_count += 1
    
    return zero_count

# Example usage



In [18]:
zeros = count_zero_scores(path)
print(f"Number of entries with score 0.0: {zeros}")

Number of entries with score 0.0: 37140


In [8]:
def copy_first_lines(src_path, dest_path, num_lines):
    """
    Copy the first `num_lines` lines from src_path to dest_path.
    
    Args:
        src_path (str): Path to the source file.
        dest_path (str): Path to the destination file.
        num_lines (int): Number of lines to copy.
    """
    with open(src_path, "r", encoding="utf-8") as src, \
         open(dest_path, "w", encoding="utf-8") as dest:
        for i, line in enumerate(src):
            if i >= num_lines:
                break
            dest.write(line)

In [11]:
src_path = "../src/results/llama/meta-llama_Llama-3.2-3B_scores copy.jsonl"
dest_path = "../src/results/meta-llama_Llama-3.2-3B_scores_46896.jsonl"
num_lines = 46896

copy_first_lines(src_path, dest_path, num_lines)

In [9]:
import random
from pathlib import Path


In [7]:
def update_validation_split(input_path, random_seed=42):
    """
    Update a JSONL file with evaluation results (EFFICIENT VERSION):
    1. Filter lines where base_eval score == 0
    2. From filtered lines, randomly select 20% and set is_validation=true
    3. Update the file in place using memory-efficient streaming
    
    Parameters
    ----------
    input_path : str | Path
        Path to the JSONL file with evaluation results to update
    random_seed : int | None
        Random seed for reproducible results. If None, uses current time.
    """
    input_path = Path(input_path)
    temp_path = input_path.with_suffix(input_path.suffix + '.tmp')
    
    # Set random seed for reproducibility
    if random_seed is not None:
        random.seed(random_seed)
    
    # PASS 1: Identify zero-score line numbers (memory efficient)
    zero_score_line_numbers = []
    total_lines = 0
    
    with input_path.open("r", encoding="utf-8") as f:
        for line_num, line in enumerate(f):
            line = line.strip()
            if line:  # Skip empty lines
                try:
                    # Only parse the fields we need for filtering
                    data = json.loads(line)
                    base_score = data.get("base_eval", {}).get("score", None)
                    if base_score == 0 or base_score == 0.0:
                        zero_score_line_numbers.append(line_num)
                    total_lines += 1
                except json.JSONDecodeError:
                    total_lines += 1
                    continue
    
    # Randomly select 20% of zero-score lines for validation
    validation_line_numbers = set()
    if zero_score_line_numbers:
        # Use round() instead of int() for better percentage accuracy
        num_validation = max(1, round(len(zero_score_line_numbers) * 0.2))
        selected_indices = random.sample(range(len(zero_score_line_numbers)), num_validation)
        validation_line_numbers = {zero_score_line_numbers[i] for i in selected_indices}
    
    # PASS 2: Stream through file and update only necessary lines
    lines_processed = 0
    validation_updates = 0
    train_updates = 0
    
    with input_path.open("r", encoding="utf-8") as input_file, \
         temp_path.open("w", encoding="utf-8") as output_file:
        
        for line_num, line in enumerate(input_file):
            line = line.strip()
            if not line:  # Skip empty lines
                output_file.write("\n")
                continue
                
            try:
                data = json.loads(line)
                
                # Check if this line needs updating
                if line_num in zero_score_line_numbers:
                    if line_num in validation_line_numbers:
                        data["is_validation"] = True
                        validation_updates += 1
                    else:
                        data["is_validation"] = False
                        train_updates += 1
                
                # Write the (possibly updated) line
                json.dump(data, output_file, ensure_ascii=False)
                output_file.write("\n")
                lines_processed += 1
                
                # Progress indicator for large files
                if lines_processed % 10000 == 0:
                    print(f"  Processed {lines_processed:,} lines...")
                    
            except json.JSONDecodeError:
                # Keep invalid lines as-is
                output_file.write(line + "\n")
                continue
    
    # Replace original file with updated version
    temp_path.replace(input_path)
    
    # Final summary
    print(f"Lines with score 0 (model eval): {len(zero_score_line_numbers)}")
    print(f"Lines with is_validation=true: {validation_updates}")


In [4]:
import json

def deduplicate_jsonl(file_path, output_path=None, key="id"):
    """
    Deduplicate a JSONL file based on a specific key (default: 'id').

    Args:
        file_path (str): Path to the input .jsonl file.
        output_path (str, optional): Path to write deduplicated output. If None, no file is written.
        key (str): The field name used to detect duplicates.

    Returns:
        list: Deduplicated list of records.
    """
    seen = set()
    deduped_records = []
    
    with open(file_path, "r", encoding="utf-8") as f:
        for ln, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                record = json.loads(line)
            except json.JSONDecodeError as e:
                print(f"❌ JSON decode error on line {ln}: {e}")
                continue
            
            ident = record.get(key)
            if ident is None:
                print(f"⚠️ Missing key '{key}' on line {ln}")
                continue

            if ident not in seen:
                seen.add(ident)
                deduped_records.append(record)

    if output_path:
        with open(output_path, "w", encoding="utf-8") as out_f:
            for rec in deduped_records:
                out_f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    print(f"✅ Deduplicated: kept {len(deduped_records)} / original {len(seen) + (len(deduped_records) - len(seen))} unique by '{key}'.")
    return deduped_records


# Example usage:



In [5]:
origin_path = "../results/meta-llama_Llama-3.2-3B_scores.jsonl"
dest_path = "../src/results/llama/meta-llama_Llama-3.2-3B_scores.jsonl"
deduped = deduplicate_jsonl(origin_path,
                            output_path=dest_path)

✅ Deduplicated: kept 76499 / original 76499 unique by 'id'.


In [10]:
update_validation_split(dest_path, random_seed=42)

  Processed 10,000 lines...
  Processed 20,000 lines...
  Processed 30,000 lines...
  Processed 40,000 lines...
  Processed 50,000 lines...
  Processed 60,000 lines...
  Processed 70,000 lines...
Lines with score 0 (model eval): 37140
Lines with is_validation=true: 7428


✅ Finished. Saved cleaned file to ../results/meta-llama_Llama-3.2-3B_scores_no_val.jsonl


In [ ]:
import json

# input and output files
infile = "../src/results/llama/meta-llama_Llama-3.2-3B_scores.jsonl"
outfile = "output.jsonl"

# keys you want to remove
keys_to_remove = [
    "meta-llama_Llama-3.2-3B_vera_r_tr100",
    "meta-llama_Llama-3.2-3B_vera_r_tr500"
]

with open(infile, "r") as fin, open(outfile, "w") as fout:
    for line in fin:
        obj = json.loads(line)
        if "ft_evals" in obj:
            for key in keys_to_remove:
                obj["ft_evals"].pop(key, None)  # remove if exists
        fout.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("Done. Cleaned file written to", outfile)
